# Première expérience — HistGradientBoostingRegressor

## Étape 1 — Charger et contrôler les données

**Objectif :** prédire la proportion de oui (`yes_share`, entre 0 et 1) pour un objet de votation dans une commune. Une observation représente une paire commune à objet.

Cette expérience utilise la table structurée existante : **analyse rétrospective sur la géographie 2026**, sans revendication de prévision historique stricte. Les limites de disponibilité des sources sont décrites dans `data/processed/modeling/README.md`.

Cette étape vérifie le contrat des données et identifie les variables candidates.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

# Retrouver la racine depuis le dossier de lancement du notebook.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())
DATA_DIR = ROOT / "data/processed/modeling"
DATA_PATH = DATA_DIR / "municipal_structured.parquet"
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
data = pd.read_parquet(DATA_PATH)
print("Source :", DATA_PATH.relative_to(ROOT))
print("SHA-256 :", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print("Mode :", manifest["mode"])

Source : data\processed\modeling\municipal_structured.parquet
SHA-256 : c8a488826093a53f307af0b4a04627fa6d78118a8ac4b25ca8c8e097fdf8e043
Mode : exploratory_retrospective


In [2]:
# La liste explicite empêche l'ajout accidentel de résultats ou d'identifiants.
TARGET = manifest["target"]
numeric_features = manifest["numeric_features"]
categorical_features = manifest["categorical_features"]
indicator_features = manifest["missing_indicators"]
feature_columns = numeric_features + categorical_features + indicator_features
required = ["municipality_id", "vote_id", "vote_date", TARGET] + feature_columns
assert not (set(required) - set(data.columns)), "Colonnes requises absentes"
assert len(feature_columns) == len(set(feature_columns)), "Prédicteurs dupliqués"
assert TARGET not in feature_columns
assert not set(feature_columns) & {"municipality_id", "vote_id", "vote_key", "vote_date"}
assert data[["municipality_id", "vote_id", "vote_date"]].notna().all().all()
assert not data.duplicated(["municipality_id", "vote_id"]).any()
assert data.groupby("vote_id")["vote_date"].nunique().eq(1).all()
assert data[TARGET].notna().all() and data[TARGET].between(0, 1).all()
assert np.isfinite(data[TARGET].to_numpy(dtype=float)).all()
assert len(data) == manifest["rows_after_joins"]
assert data["vote_id"].nunique() == manifest["objects"]
assert data["vote_date"].nunique() == manifest["dates"]

summary = pd.Series({
    "Lignes": len(data),
    "Communes": data["municipality_id"].nunique(),
    "Objets": data["vote_id"].nunique(),
    "Dates": data["vote_date"].nunique(),
    "Première date": data["vote_date"].min().date(),
    "Dernière date": data["vote_date"].max().date(),
    "Variables numériques candidates": len(numeric_features),
    "Variables catégorielles candidates": len(categorical_features),
    "Indicateurs de valeurs manquantes": len(indicator_features),
}, name="Valeur")
display(summary.to_frame())
print("Contrôles de structure et de cible réussis.")

Contrôles de structure et de cible réussis.


,Valeur
Lignes,262981
Communes,2105
Objets,125
Dates,44
Première date,2012-03-11
Dernière date,2025-09-28
Variables numériques candidates,19
Variables catégorielles candidates,6
Indicateurs de valeurs manquantes,3


In [3]:
# Couverture annuelle pour préparer un découpage par dates entières.
coverage = (
    data.assign(year=data["vote_date"].dt.year)
    .groupby("year")
    .agg(rows=("vote_id", "size"), objects=("vote_id", "nunique"),
         dates=("vote_date", "nunique"), municipalities=("municipality_id", "nunique"))
)
display(coverage)

feature_audit = pd.DataFrame({
    "dtype": data[feature_columns].dtypes.astype(str),
    "missing_rows": data[feature_columns].isna().sum(),
    "missing_pct": data[feature_columns].isna().mean().mul(100).round(2),
    "unique_values": data[feature_columns].nunique(),
})
feature_audit["role"] = [
    "numeric" if c in numeric_features else "categorical" if c in categorical_features else "indicator"
    for c in feature_audit.index
]
display(feature_audit.sort_values("missing_pct", ascending=False))

,rows,objects,dates,municipalities
year,,,,
2012,25236,12,4,2103
2013,23133,11,4,2103
2014,25236,12,4,2103
2015,12618,6,2,2103
2016,27339,13,4,2103
2017,14721,7,3,2103
2018,21040,10,4,2104
2019,6312,3,2,2104
2020,18936,9,3,2104


,dtype,missing_rows,missing_pct,unique_values,role
party_share_centre,Float64,231511,88.03,2096,numeric
object_domain_3,string,199849,75.99,10,categorical
party_share_pbd,Float64,75046,28.54,5016,numeric
object_domain_2,string,63108,24.00,12,categorical
party_share_pev,Float64,56983,21.67,6420,numeric
party_share_pdc,Float64,36294,13.80,6129,numeric
party_share_pvl,Float64,30931,11.76,7413,numeric
party_share_verts,Float64,10525,4.00,7981,numeric
party_share_ps,Float64,4865,1.85,8170,numeric
party_share_plr,Float64,3913,1.49,8210,numeric


La prochaine étape sera de fixer les ensembles entraînement, validation et test **chronologiquement, par dates entières** : toutes les communes et tous les objets d'une même date restent ensemble. Le test sera réservé à l'évaluation finale.

Les variables `history_mean_yes`, `history_n_objects` et `history_missing` sont candidates : leur historique devra être recalculé selon le protocole choisi (figé ou roulant), avant l'entraînement. La table contient actuellement un historique roulant sur les dates antérieures.

Les codes de domaines et de typologie sont catégoriels, même lorsqu'ils ressemblent à des nombres. Les valeurs absentes des partis ne seront pas assimilées à des zéros. L'encodage et les éventuels prétraitements seront ajustés uniquement sur l'entraînement.

## Étape 2 — Partitions chronologiques et historique figé

Les bornes sont choisies sur la couverture, avant tout calcul de performance :

- **Entraînement : 2012–2020**, pour apprendre sur la période la plus ancienne.
- **Validation : 2021–2022**, pour comparer les choix du modèle.
- **Test : 2023–2025**, réservé à une évaluation finale après fixation des choix.

Chaque date entière appartient à une seule partition. L'historique d'une ligne d'entraînement utilise uniquement les dates précédentes, y compris avant 2012. Pour la validation et le test, il est **figé à fin 2020** : aucun résultat de validation ou de test n'alimente les prédicteurs. Ce protocole n'inclut pas de réentraînement sur train + validation. Les dates de publication historiques restent non vérifiées : le cadre demeure rétrospectif.

In [4]:
TRAIN_END = pd.Timestamp("2020-12-31")
VALIDATION_END = pd.Timestamp("2022-12-31")
SEED = 42

experiment_data = data.copy()
experiment_data["partition"] = np.select(
    [experiment_data.vote_date.le(TRAIN_END), experiment_data.vote_date.le(VALIDATION_END)],
    ["train", "validation"], default="test",
)
partition_order = ["train", "validation", "test"]
partition_summary = experiment_data.groupby("partition").agg(
    first_date=("vote_date", "min"), last_date=("vote_date", "max"),
    rows=("vote_id", "size"), objects=("vote_id", "nunique"), dates=("vote_date", "nunique"),
).reindex(partition_order)
assert partition_summary.rows.gt(0).all()
assert experiment_data.groupby("vote_date").partition.nunique().eq(1).all()
assert experiment_data.groupby("vote_id").partition.nunique().eq(1).all()
assert partition_summary.loc["train", "last_date"] < partition_summary.loc["validation", "first_date"]
assert partition_summary.loc["validation", "last_date"] < partition_summary.loc["test", "first_date"]
display(partition_summary)

,first_date,last_date,rows,objects,dates
partition,,,,,
train,2012-03-11,2020-11-29,174571,83,30
validation,2021-03-07,2022-09-25,50520,24,7
test,2023-06-18,2025-09-28,37890,18,7


In [5]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.integration.build_modeling_dataset import clean_outcomes

# Réutiliser exactement les exclusions du constructeur de la table.
history_path = ROOT / "data/processed/ofs-votes/votes_communes.parquet"
history_sha256 = hashlib.sha256(history_path.read_bytes()).hexdigest()
assert history_sha256 == manifest["sources"]["ofs"]["sha256"], "Source OFS modifiée : reconstruire la table"
history_source, _ = clean_outcomes(pd.read_parquet(history_path))

def frozen_history(targets, outcomes, train_end):
    # Seuls les résultats de la période autorisée peuvent entrer dans l'historique.
    pool = outcomes.loc[outcomes.vote_date.le(train_end)]
    result = pd.DataFrame(index=targets.index, columns=[
        "history_mean_yes", "history_n_objects", "history_last_vote_date"
    ])
    for date, group in targets.groupby("vote_date", sort=True):
        past = pool.loc[pool.vote_date.lt(date)]
        stats = past.groupby("municipality_id").agg(
            history_mean_yes=("yes_share", "mean"),
            history_n_objects=("yes_share", "count"),
            history_last_vote_date=("vote_date", "max"),
        )
        for column in result.columns:
            result.loc[group.index, column] = group.municipality_id.map(stats[column]).to_numpy()
    result["history_mean_yes"] = pd.to_numeric(result.history_mean_yes)
    result["history_n_objects"] = pd.to_numeric(result.history_n_objects).fillna(0).astype("int64")
    result["history_last_vote_date"] = pd.to_datetime(result.history_last_vote_date)
    result["history_missing"] = result.history_mean_yes.isna()
    return result

history = frozen_history(experiment_data, history_source, TRAIN_END)
experiment_data[history.columns] = history
known = experiment_data.history_last_vote_date.notna()
assert (experiment_data.loc[known, "history_last_vote_date"] < experiment_data.loc[known, "vote_date"]).all()
assert experiment_data.loc[known, "history_last_vote_date"].le(TRAIN_END).all()

# L'historique d'entraînement doit retrouver celui de la table initiale.
train_mask = experiment_data.partition.eq("train")
np.testing.assert_allclose(
    experiment_data.loc[train_mask, "history_mean_yes"].to_numpy(dtype=float, na_value=np.nan),
    data.loc[train_mask, "history_mean_yes"].to_numpy(dtype=float, na_value=np.nan), equal_nan=True,
)
display(experiment_data.groupby("partition").agg(
    missing_history=("history_missing", "sum"),
    last_history_date=("history_last_vote_date", "max"),
).reindex(partition_order))

,missing_history,last_history_date
partition,,
train,2,2020-09-27
validation,24,2020-11-29
test,18,2020-11-29


In [6]:
# Petit contrôle indépendant : même jour exclu, résultats réservés sans effet.
toy_outcomes = pd.DataFrame({
    "municipality_id": ["a", "a", "a", "a"],
    "vote_date": pd.to_datetime(["2019-01-01", "2020-01-01", "2020-01-01", "2021-01-01"]),
    "yes_share": [0.2, 0.4, 0.8, 1.0],
})
toy_targets = pd.DataFrame({
    "municipality_id": ["a", "a", "a", "new"],
    "vote_date": pd.to_datetime(["2020-01-01", "2021-01-01", "2023-01-01", "2023-01-01"]),
})
toy_history = frozen_history(toy_targets, toy_outcomes, TRAIN_END)
np.testing.assert_allclose(toy_history.history_mean_yes, [0.2, 1.4 / 3, 1.4 / 3, np.nan], equal_nan=True)
assert toy_history.history_n_objects.tolist() == [1, 3, 3, 0]
changed = toy_outcomes.copy()
changed.loc[changed.vote_date.gt(TRAIN_END), "yes_share"] = 0.0
pd.testing.assert_frame_equal(toy_history, frozen_history(toy_targets, changed, TRAIN_END))
print("Contrôles réussis : dates séparées, historique passé et figé, résultats réservés sans effet.")

Contrôles réussis : dates séparées, historique passé et figé, résultats réservés sans effet.


In [7]:
# Préparer les matrices sans ajuster de prétraitement et sans examiner les cibles test.
X_train = experiment_data.loc[experiment_data.partition.eq("train"), feature_columns].copy()
y_train = experiment_data.loc[X_train.index, TARGET].copy()
X_validation = experiment_data.loc[experiment_data.partition.eq("validation"), feature_columns].copy()
y_validation = experiment_data.loc[X_validation.index, TARGET].copy()
test_index = experiment_data.index[experiment_data.partition.eq("test")]

protocol = {
    "model": "HistGradientBoostingRegressor",
    "mode": "exploratory_retrospective",
    "seed": SEED,
    "dataset_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "history_source_sha256": history_sha256,
    "train_end": str(TRAIN_END.date()),
    "validation_end": str(VALIDATION_END.date()),
    "history_policy": "strictly earlier dates; frozen at train_end; pre-2012 outcomes included",
    "refit_train_validation": False,
    "metric": "MAE in percentage points, equal weight per municipality-object row",
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "indicator_features": indicator_features,
    "partition_dates": {
        part: sorted(experiment_data.loc[experiment_data.partition.eq(part), "vote_date"].dt.strftime("%Y-%m-%d").unique().tolist())
        for part in partition_order
    },
}
output_dir = ROOT / "reports/generated/hist_gradient_boosting"
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8")
partition_summary.to_csv(output_dir / "partitions.csv")
print("Train :", X_train.shape, "; validation :", X_validation.shape, "; lignes test réservées :", len(test_index))
print("Protocole enregistré dans", output_dir.relative_to(ROOT))

Train : (174571, 28) ; validation : (50520, 28) ; lignes test réservées : 37890
Protocole enregistré dans reports\generated\hist_gradient_boosting


La prochaine étape consiste à entraîner les baselines et un premier `HistGradientBoostingRegressor`, puis à comparer leurs MAE sur la validation uniquement. Le test reste réservé.

## Étape 3 — Premier entraînement et comparaison sur la validation

Trois prédictions sont comparées sur exactement les mêmes lignes :

- **Constante** : moyenne de `yes_share` sur les lignes d'entraînement.
- **Historique communal** : moyenne historique figée fin 2020 ; repli sur la constante pour une commune sans historique.
- **HistGradientBoostingRegressor** : les 28 prédicteurs retenus, avec paramètres fixés avant de mesurer les scores.

Premier réglage : perte quadratique, 100 itérations, taux d'apprentissage 0,1, au plus 31 feuilles et au moins 20 observations par feuille. L'arrêt anticipé est désactivé pour éviter une séparation interne aléatoire. Aucune recherche d'hyperparamètres n'est effectuée.

Les catégories sont encodées sur l'entraînement uniquement et déclarées catégorielles au modèle. Les catégories inconnues deviennent manquantes. Le modèle gère directement les valeurs manquantes ; aucune normalisation ni imputation n'est appliquée. Les prédictions sont bornées entre 0 et 1.

La MAE principale donne le même poids à chaque ligne commune × objet. Les MAE moyennes par objet et par date sont aussi présentées pour éclairer cette pondération. Tous les scores portent sur **2021–2022**, jamais sur le test.

In [8]:
from time import perf_counter
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

def model_input(frame):
    # Harmoniser les types pandas sans apprendre de statistique sur les données.
    result = frame.copy()
    for column in numeric_features + indicator_features:
        result[column] = result[column].astype(float)
    for column in categorical_features:
        result[column] = result[column].astype(object).where(result[column].notna(), np.nan)
    return result

train_input = model_input(X_train)
validation_input = model_input(X_validation)
assert train_input[categorical_features].nunique().le(255).all()
empty_training_features = train_input.columns[train_input.isna().all()].tolist()
continuous_columns = [c for c in numeric_features + indicator_features if c not in empty_training_features]
active_categorical = [c for c in categorical_features if c not in empty_training_features]
print("Excluded all-missing training columns:", empty_training_features)
preprocessor = ColumnTransformer([
    ("numeric", "passthrough", continuous_columns),
    ("categorical", OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan,
        encoded_missing_value=np.nan, dtype=float,
    ), active_categorical),
], sparse_threshold=0)
settings = dict(
    loss="squared_error", learning_rate=0.1, max_iter=100,
    max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=0.0,
    early_stopping=False, random_state=SEED,
)
model = Pipeline([
    ("prepare", preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        **settings,
        categorical_features=[False] * len(continuous_columns) + [True] * len(active_categorical),
    )),
])
started = perf_counter()
model.fit(train_input, y_train.astype(float))
training_seconds = perf_counter() - started
raw_prediction = model.predict(validation_input)
assert np.isfinite(raw_prediction).all()
print("scikit-learn :", sklearn.__version__)
print("Itérations :", model.named_steps["regressor"].n_iter_)
print(f"Durée d'entraînement : {training_seconds:.2f} s")

Excluded all-missing training columns: ['party_share_centre']
scikit-learn : 1.9.1
Itérations : 100
Durée d'entraînement : 3.63 s


In [9]:
constant_prediction = float(y_train.mean())
predictions = {
    "constant": np.full(len(y_validation), constant_prediction),
    "municipal_history": X_validation.history_mean_yes.fillna(constant_prediction).to_numpy(dtype=float),
    "hist_gradient_boosting": np.clip(raw_prediction, 0, 1),
}
validation_predictions = experiment_data.loc[
    X_validation.index, ["municipality_id", "vote_id", "vote_date", TARGET]
].copy()
score_rows = []
object_tables = []
date_tables = []
for name, prediction in predictions.items():
    assert len(prediction) == len(y_validation) and np.isfinite(prediction).all()
    assert ((prediction >= 0) & (prediction <= 1)).all()
    validation_predictions[name] = prediction
    errors = validation_predictions[["vote_id", "vote_date"]].copy()
    errors["absolute_error_pp"] = np.abs(y_validation.to_numpy(dtype=float) - prediction) * 100
    by_object = errors.groupby(["vote_id", "vote_date"], as_index=False).absolute_error_pp.mean()
    by_date = errors.groupby("vote_date", as_index=False).absolute_error_pp.mean()
    object_tables.append(by_object.assign(model=name))
    date_tables.append(by_date.assign(model=name))
    score_rows.append({
        "model": name,
        "mae_pp": errors.absolute_error_pp.mean(),
        "mean_object_mae_pp": by_object.absolute_error_pp.mean(),
        "mean_date_mae_pp": by_date.absolute_error_pp.mean(),
    })
scores = pd.DataFrame(score_rows).set_index("model")
scores["gain_vs_constant_pp"] = scores.loc["constant", "mae_pp"] - scores.mae_pp
scores["gain_vs_history_pp"] = scores.loc["municipal_history", "mae_pp"] - scores.mae_pp
display(scores.round(3))
object_scores = pd.concat(object_tables, ignore_index=True)
date_scores = pd.concat(date_tables, ignore_index=True)
display(date_scores.pivot(index="vote_date", columns="model", values="absolute_error_pp").round(3))
print("Replis de la baseline historique :", int(X_validation.history_mean_yes.isna().sum()))
print("Prédictions boosting bornées :", int(((raw_prediction < 0) | (raw_prediction > 1)).sum()))

Replis de la baseline historique : 24
Prédictions boosting bornées : 0


,mae_pp,mean_object_mae_pp,mean_date_mae_pp,gain_vs_constant_pp,gain_vs_history_pp
model,,,,,
constant,13.173,13.173,13.388,0.000,-0.203
municipal_history,12.970,12.970,13.143,0.203,0.000
hist_gradient_boosting,12.005,12.005,12.345,1.167,0.965


model,constant,hist_gradient_boosting,municipal_history
vote_date,,,
2021-03-07,10.571,17.133,11.000
2021-06-13,12.784,8.612,12.498
2021-09-26,14.961,11.526,14.813
2021-11-28,13.856,12.719,13.220
2022-02-13,13.767,10.602,13.973
2022-05-15,17.166,13.242,15.364
2022-09-25,10.613,12.582,11.129


In [10]:
import platform
import joblib

run_manifest = {
    **protocol,
    "stage": "first_validation_run",
    "thread_limit": 1,
    "excluded_all_missing_training_features": empty_training_features,
    "effective_features": continuous_columns + active_categorical,
    "settings": settings,
    "iterations": int(model.named_steps["regressor"].n_iter_),
    "training_seconds": training_seconds,
    "training_rows": len(X_train),
    "validation_rows": len(X_validation),
    "validation_objects": validation_predictions.vote_id.nunique(),
    "validation_dates": validation_predictions.vote_date.nunique(),
    "constant_prediction": constant_prediction,
    "prediction_clipping": [0, 1],
    "clipped_validation_predictions": int(((raw_prediction < 0) | (raw_prediction > 1)).sum()),
    "preprocessing": "training-only ordinal encoding, native categorical splits and missing values; no imputation or scaling",
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "test_evaluated": False,
}
scores.to_csv(output_dir / "validation_metrics.csv")
validation_predictions.to_parquet(output_dir / "validation_predictions.parquet", index=False)
object_scores.to_csv(output_dir / "validation_by_object.csv", index=False)
date_scores.to_csv(output_dir / "validation_by_date.csv", index=False)
(output_dir / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
model_dir = ROOT / "models/hist_gradient_boosting"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": model, "features": feature_columns, "manifest": run_manifest}, model_dir / "first_validation.joblib")
print("Résultats :", output_dir.relative_to(ROOT))
print("Modèle :", (model_dir / "first_validation.joblib").relative_to(ROOT))
print("Test réservé : aucun score calculé.")

Résultats : reports\generated\hist_gradient_boosting
Modèle : models\hist_gradient_boosting\first_validation.joblib
Test réservé : aucun score calculé.


**Lecture des résultats :** une MAE de 10 points signifie un écart absolu moyen de 10 points de pourcentage sur la proportion de oui. Un gain positif indique une amélioration sur la baseline correspondante.

Il s'agit d'une première mesure de validation rétrospective, avec seulement 24 objets sur 7 dates. Les nombreuses communes d'un objet ne constituent pas autant de votations indépendantes. Les résultats ne démontrent donc pas, à eux seuls, une capacité de prévision historique stricte ou de généralisation sur le test.

### Premier résultat observé

| Modèle | MAE validation (points de pourcentage) |
|---|---:|
| Constante | 13,173 |
| Historique communal | 12,970 |
| HistGradientBoostingRegressor | 12,005 |

Le boosting gagne **0,965 point** sur l'historique communal (environ **7,4 %** de baisse de MAE) et **1,167 point** sur la constante. Ces scores portent sur les 50 520 lignes de validation, 24 objets et 7 dates.

Les 100 itérations ont été ajustées sur 174 571 lignes de 2012–2020. Parmi les 28 variables candidates, `party_share_centre` est entièrement absente de l'entraînement : elle est exclue avant ajustement, ce qui laisse **27 prédicteurs effectifs**. Ce filtre utilise uniquement l'entraînement. Les autres valeurs manquantes restent gérées par le modèle.

Exécuter le notebook depuis un noyau neuf : `OMP_NUM_THREADS=1` est fixé avant les imports pour ce premier essai sous Windows. Les cellules ont été exécutées dans IPython avec l'environnement `swiss-votes-ml` ; les sorties sont conservées. Le test 2023–2025 reste non évalué.